In [102]:
from ruwordnet import RuWordNet
import pymorphy3
import re

wn = RuWordNet()
morph = pymorphy3.MorphAnalyzer()

In [103]:
def preprocess(text):
    import re
    words = re.findall(r'\w+', text.lower())
    
    lemmas = []
    for w in words:
        p = morph.parse(w)[0]
        
        # фильтр частей речи
        if p.tag.POS not in {'PREP', 'CONJ', 'PRCL'}:
            lemmas.append(p.normal_form)

    return set(lemmas)

In [104]:
def expanded(words):
    expanded_set = set(words)

    for w in words:
        senses = wn.get_senses(w)
        
        for sense in senses:
            synset = sense.synset

            for s in synset.senses:
                lemmas = preprocess(s.name)
                expanded_set.update(lemmas)

            for h in synset.hypernyms[:1]:
                for s in h.senses:
                    lemmas = preprocess(s.name)
                    expanded_set.update(lemmas)

            for h in synset.hyponyms[:1]:
                for s in h.senses:
                    lemmas = preprocess(s.name)
                    expanded_set.update(lemmas)

    return expanded_set

In [105]:
def lesk(sentence, word):
    context = preprocess(sentence) - preprocess(word)
    context = expanded(context)
    print(f"Context: {context}")
    
    senses = wn.get_senses(word)
    
    best_synset = (None, None)
    best_score = 0

    for sense in senses:
        synset = sense.synset
        
        text = synset.definition or ""
        
        for s in synset.senses:
            text += " " + s.name
        
        for h in synset.hypernyms[:1]:
            for s in h.senses:
                text += " " + s.name

        for h in synset.hyponyms[:1]:
            for s in h.senses:
                text += " " + s.name

        signature = preprocess(text)
        
        intersection = len(context & signature)

        if len(signature) == 0:
            score = 0
        else:
            score = intersection / len(signature)
        
        if score > best_score:
            best_score = score 
            best_synset = (synset, text)
            
        print(f"Synset: {synset}, Score: {score:.4f}, Signature: {signature}")

    return best_synset

In [106]:
def get_relations(synset):
    synonyms = []
    for sense in synset.senses:
        synonyms.append(sense.name)

    hypernyms = []
    for h in synset.hypernyms[:1]:
        for s in h.senses:
            hypernyms.append(s.name)

    hyponyms = []
    for h in synset.hyponyms[:1]:
        hyponyms.append(h.title)

    return synonyms, hypernyms, hyponyms

In [107]:
sentence = "Я получил хорошую оценку за экзамен"
word = "оценка"

synset, synset_text = lesk(sentence, word)

print("Выбранное значение:")
print(synset_text)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Context: {'сделать', 'крючок', 'распоряжение', 'созидать', 'обмануться', 'совершать', 'острый', 'качественный', 'акт', 'воздействие', 'созидаться', 'обман', 'сдача', 'заиметь', 'делаться', 'экзаменационный', 'овладевать', 'тест', 'получать', 'насоздавать', 'синтезировать', 'получить', 'создавать', 'совершить', 'подпасть', 'повестись', 'госэкзамен', 'проверка', 'подвергаться', 'совершение', 'попасться', 'создать', 'обманываться', 'остаться', 'тонкий', 'подпадать', 'учебный', 'стать', 'делать', 'отношение', 'целенаправленный', 'испытание', 'задание', 'положительный', 'хитрость', 'овладеть', 'испытать', 'хороший', 'состоять', 'я', 'подвергнуться', 'испытывать', 'изощрённый', 'нос', 'свой', 'вещество', 'предмет', 'попадаться', 'удочка', 'экзамен', 'государственный', 'действие', 'уловка', 'действовать'}
Synset: Synset(id="115032-N", title="ОЦЕНИТЬ, ОПРЕДЕЛИТЬ ЦЕНУ"), Score: 0.0000, Signature: {'установить', 'установление', 'переоценивание', 'определение', 'задавание', 'цена', 'что', 'опреде

In [108]:
sentence = "я ключом открыл дверь"
word = "ключ"

synset, synset_text = lesk(sentence, word)

print("Выбранное значение:")
print(synset_text)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Context: {'дверца', 'открываться', 'доска', 'совершать', 'класть', 'растворить', 'причина', 'плита', 'дверь', 'толчок', 'раскрывать', 'отворяться', 'оставлять', 'совершить', 'крышка', 'устанавливаться', 'приоткрыть', 'коробка', 'способствовать', 'приотворять', 'определить', 'зачинать', 'приотворить', 'явиться', 'я', 'начало', 'оголять', 'произвести', 'оголить', 'обнажать', 'начинать', 'выяснять', 'нападение', 'открыть', 'проесть', 'определять', 'пооткрывать', 'производить', 'переместить', 'дверной', 'послужить', 'довыяснить', 'открыться', 'действие', 'разворачивать', 'открывать', 'сделать', 'выяснить', 'отворить', 'конструкция', 'раскрыть', 'ворота', 'оставить', 'блок', 'доступ', 'поднять', 'незащищённый', 'делаться', 'врата', 'давать', 'развернуть', 'развёртывать', 'положить', 'делать', 'строительный', 'обусловливать', 'обусловить', 'подымать', 'устроить', 'элемент', 'открытие', 'установить', 'отворять', 'приоткрывать', 'перемещать', 'стена', 'отвориться', 'дверка', 'поднимать', 'уста

In [109]:
sentence = "я ключом открыл замок"
word = "ключ"

synset, synset_text = lesk(sentence, word)

print("Выбранное значение:")
print(synset_text)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Context: {'открываться', 'совершать', 'класть', 'растворить', 'причина', 'кодовый', 'механизм', 'толчок', 'раскрывать', 'отворяться', 'оставлять', 'наборный', 'совершить', 'крышка', 'устанавливаться', 'приоткрыть', 'способствовать', 'приотворять', 'определить', 'зачинать', 'приотворить', 'затворный', 'я', 'палаццо', 'явиться', 'запорный', 'оголять', 'начало', 'произвести', 'оголить', 'обнажать', 'начинать', 'запирание', 'нападение', 'выяснять', 'открыть', 'определять', 'пооткрывать', 'производить', 'переместить', 'замочка', 'послужить', 'феодал', 'средневековый', 'довыяснить', 'открыться', 'действие', 'разворачивать', 'код', 'комплекс', 'открывать', 'сделать', 'выяснить', 'отворить', 'замок', 'запирать', 'раскрыть', 'запор', 'оставить', 'доступ', 'поднять', 'незащищённый', 'делаться', 'давать', 'развернуть', 'развёртывать', 'положить', 'шифр', 'делать', 'обусловливать', 'обусловить', 'замковый', 'подымать', 'устройство', 'устроить', 'открытие', 'установить', 'дворец', 'отворять', 'прио

In [110]:
senses = wn.get_senses("ключ")

for sense in senses:
    synset = sense.synset
    print(synset.definition)

знак в начале нотной строки, определяющий привязку нот к той или иной высоте звука







In [111]:
senses = wn.get_senses("оценка")

for sense in senses:
    synset = sense.synset
    print(synset.definition)

определить, установить стоимость, цену, ценность чего-либо
отметка, выставляемая в учебных заведениях
мнение, суждение или отзыв о чём-либо
мнение, суждение или отзыв о чём-либо


In [112]:
senses = wn.get_senses("ручка")

for sense in senses:
    synset = sense.synset
    print(synset.definition)

письменная принадлежность в виде удлиненного стержня, на конце которого находится перо или другое пишущее приспособление
одна из двух верхних конечностей человека, от плеча до конца пальцев
часть предмета, за которую его можно брать рукой, держать или перемещать

опора для рук (локтей) на стуле c подлокотниками или мягкой мебелиподлокотник кресла


In [113]:
sentence = "возьми письменную ручку"
word = "ручка"

synset, synset_text = lesk(sentence, word)

print("Выбранное значение:")
print(synset_text)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Context: {'взять', 'сделать', 'распоряжение', 'число', 'счесть', 'избирать', 'унести', 'уноситься', 'отбираться', 'набирать', 'качественный', 'удалить', 'тронуть', 'доставать', 'заиметь', 'получать', 'получить', 'отбирать', 'остановиться', 'письменный', 'себя', 'выбор', 'определиться', 'останавливать', 'извлечь', 'избрать', 'останавливаться', 'делать', 'уносить', 'потрогать', 'хороший', 'пасть', 'отобраться', 'отобрать', 'притронуться', 'трогать', 'прикоснуться', 'свой', 'выбрать', 'притрагиваться', 'посчитать', 'захватывать', 'прихватывать', 'удалять', 'написать', 'извлекать', 'отдалить', 'остановить', 'отдалять', 'достать', 'прикасаться', 'браться', 'прихватить', 'брать', 'захватить', 'набрать', 'выбирать'}
Synset: Synset(id="110938-N", title="РУЧКА ДЛЯ ПИСЬМА"), Score: 0.0625, Signature: {'авторучка', 'стержень', 'принадлежность', 'удлинённый', 'который', 'другой', 'перо', 'находиться', 'ручка', 'вид', 'пишущий', 'автоматический', 'письмо', 'письменный', 'приспособление', 'конец'}
S

In [114]:
sentence = "зажглась звезда на небе"
word = "звезда"

synset, synset_text = lesk(sentence, word)

print("Выбранное значение:")
print(synset_text)

synonyms, hypernyms, hyponyms = get_relations(synset)

print("\nСинонимы:", synonyms)
print("Гиперонимы:", hypernyms)
print("Гипонимы:", hyponyms)

Context: {'самовоспламеняться', 'небо', 'класть', 'просиять', 'зажигаться', 'вспыхнуть', 'возгораться', 'загораться', 'пространство', 'свет', 'воздух', 'разгораться', 'самовоспламениться', 'орган', 'рот', 'давать', 'полого', 'положить', 'воспламеняться', 'гореть', 'небосклон', 'самовозгореться', 'засветиться', 'засветить', 'небосвод', 'разжигаться', 'засиять', 'стенка', 'зачинать', 'разжечься', 'пламя', 'купол', 'зажечься', 'начало', 'вспыхивать', 'воспламениться', 'начинать', 'загореться', 'разгореться', 'самовозгораться', 'возгореться', 'небесный', 'светиться', 'свод', 'воздушный', 'полость', 'начать'}
Synset: Synset(id="155137-N", title="ЗВЕЗДНОСТЬ ГОСТИНИЦЫ"), Score: 0.0000, Signature: {'категория', 'отель', 'оценка', 'звезда', 'характеристика', 'гостиница', 'оценочный', 'звёздность'}
Synset: Synset(id="109931-N", title="ЗВЕЗДА (НЕБЕСНОЕ ТЕЛО)"), Score: 0.1053, Signature: {'светящийся', 'солнце', 'звезда', 'видимый', 'светило', 'огромный', 'диск', 'природа', 'солнышко', 'свой', 'те